# CHURN PREDICTION MODEL


### Problem Statement
##### Customers leaving our business costs us money. We need a system to predict which customers are likely to leave soon so we can try to keep them.

## 1. Initializing the project


#### importing the required packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import  LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 2. Data Collection

In [ ]:
df = pd.read_csv('./data.csv')

## 3. Data Exploration and Pre-processing

#### 3.1 Data Inspection

In [ ]:
df.head()              # View top 5 rows
# df.tail()              # View bottom 5 rows

In [ ]:
df.shape               # Rows and columns

In [ ]:
df.columns             # Column names             

In [ ]:
df.dtypes              # Data types of columns 

In [ ]:
df.info()              # Full summary (nulls, dtypes, etc.)

In [ ]:
df.describe()          # Summary statistics for numerical columns               

In [ ]:
df.nunique()           # Count of unique values per column

#### 3.2 Missing Data Handling

In [ ]:
df.isnull().sum()                       # Count nulls per column

In [ ]:
df.dropna()                           # Drop rows with any null values


In [ ]:
value = 'any value you want to replace the nulls with'
df.fillna(value)                        # Fill nulls with a specific value

In [ ]:
df['actual_col_name'].fillna(df['col'].mean())     # Fill with mean/median/mode

In [ ]:
df.interpolate()                        # Linear interpolation for missing values

#### 3.3 Outlier Detection & Treatment

In [ ]:
# # Using IQR
# Q1 = df['col'].quantile(0.25)
# Q3 = df['col'].quantile(0.75)
# IQR = Q3 - Q1
# outliers = df[(df['col'] < Q1 - 1.5 * IQR) | (df['col'] > Q3 + 1.5 * IQR)]

# # Z-score
# from scipy.stats import zscore
# df['z_score'] = zscore(df['col'])

# # Remove outliers
# df = df[(df['col'] >= lower_limit) & (df['col'] <= upper_limit)]


#### 3.4 Duplicates Handling

In [ ]:
df[df.duplicated()]
df.drop_duplicates(inplace=True)

#### 3.5 Type Conversion

In [ ]:
# df['col'] = df['col'].astype(int)
# df['date'] = pd.to_datetime(df['date']) 

#### 3.6 Categorical Encoding

In [ ]:
# pd.get_dummies(df['col'])         # One-hot encoding
# df.replace({'Yes': 1, 'No': 0})   # Manual binary encoding
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df['col'] = le.fit_transform(df['col'])


label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])
df= pd.get_dummies(df, columns=['Geography'], drop_first=True)

In [ ]:
df

#### 3.7 Feature Selection


In [ ]:
features = ['CreditScore', 'Gender' , 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain']
X = df[features]
y = df['Exited']

In [ ]:
#### Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### 3.8 Feature Scaling / Normalization

In [ ]:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# scaler = StandardScaler()
# df[['col1', 'col2']] = scaler.fit_transform(df[['col1', 'col2']])

# # Other options
# MinMaxScaler()      # Scales to [0,1]
# RobustScaler()      # Less sensitive to outliers



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


#### 3.9 Feature Extraction

In [ ]:
# # From datetime
# df['year'] = df['date'].dt.year
# df['month'] = df['date'].dt.month
# df['weekday'] = df['date'].dt.weekday

# # Text-based
# df['text_length'] = df['text'].apply(len)
# df['word_count'] = df['text'].apply(lambda x: len(x.split()))


#### 3.10 Visualization for EDA

In [ ]:
# df['col'].value_counts().plot(kind='bar')        # Count plot
# df['col'].hist()                                  # Histogram
# sns.boxplot(x='col', data=df)                     # Boxplot
# sns.heatmap(df.corr(), annot=True)                # Correlation heatmap
# sns.pairplot(df)                                   # Pairwise relationships


#### 3.11 Correlation Analysis

In [ ]:
# df.corr()                  # Pearson correlation between features
# sns.heatmap(df.corr())     # Visualize correlation


#### 3.12 Handling Skewness

In [ ]:
# df['log_col'] = np.log1p(df['col'])               # Log transform
# df['sqrt_col'] = np.sqrt(df['col'])               # Square root
# df['box_cox_col'], _ = stats.boxcox(df['col'])    # Box-Cox transform

## 4. Visual representation of the Data

#### 4.1 Churn Count (Exited = 0 or 1)

In [ ]:
sns.countplot(x='Exited', data=df)
plt.title('Churn Count')
plt.xticks([0, 1], ['Stayed', 'Exited'])
plt.ylabel('Number of Customers')
plt.show()

#### 4.2 Gender vs Churn

In [ ]:
sns.countplot(x='Gender', hue='Exited', data=df)
plt.title('Gender vs Churn')
plt.legend(title='Exited', labels=['No', 'Yes'])
plt.show()

#### 4.3 Geography vs Churn

In [ ]:
sns.countplot(x='Geography', hue='Exited', data=df)
plt.title('Geography vs Churn')
plt.ylabel('Number of Customers')
plt.show()

#### 4.4 Age Distribution

In [ ]:
sns.histplot(df['Age'], kde=True, bins=30)
plt.title('Age Distribution of Customers')
plt.show()

#### 4.5 Age vs Exited (Boxplot)

In [ ]:
sns.boxplot(x='Exited', y='Age', data=df)
plt.title('Age Distribution by Churn')
plt.xticks([0, 1], ['Stayed', 'Exited'])
plt.show()

#### 4.6 Credit Score Distribution

In [ ]:
sns.histplot(df['CreditScore'], kde=True, bins=30)
plt.title('Credit Score Distribution')
plt.show()

#### 4.7 Number of Products vs Churn

In [ ]:
sns.countplot(x='NumOfProducts', hue='Exited', data=df)
plt.title('Number of Products vs Churn')
plt.show()

#### 4.8 Correlation Heatmap (Numerical Only)

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

#### 4.9 Estimated Salary Distribution

In [ ]:
sns.histplot(df['EstimatedSalary'], kde=True, bins=30)
plt.title('Estimated Salary Distribution')
plt.show()

#### 4.10 Balance vs Exited (Boxplot)

In [ ]:
sns.boxplot(x='Exited', y='Balance', data=df)
plt.title('Balance vs Churn')
plt.xticks([0, 1], ['Stayed', 'Exited'])
plt.show()

## 5. Model Training & Tuning & Evaluation

In [ ]:
# model = RandomForestClassifier(n_estimators=100, random_state=42)
# model.fit(X_train, y_train)



# Model Evaluation
def model_evaluation(true, predicted):
    conf_matrix = confusion_matrix(true, predicted)
    class_report = classification_report(true, predicted)
    accuracy = accuracy_score(true, predicted)
    return conf_matrix, class_report, accuracy


models_list = []
accuracy_scores_list = []


# Model selection
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42),                                  # SVM with RBF kernel has higher accuracy
    # 'SVM': SVC(kernel='linear', class_weight='balanced', random_state=42),    # SVM with linear kernel and balanced class weights has low accuracy
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for i in range(len(list(models))):
    # Model training
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    
    # Model predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    
    # Evaluation of train and test dataset
    train_conf_matrix, train_class_report, train_accuracy = model_evaluation(y_train, y_train_pred)
    test_conf_matrix, test_class_report, test_accuracy = model_evaluation(y_test, y_test_pred)
    
    print(f"Model: {list(models.keys())[i]}")
    models_list.append(list(models.keys())[i])
    
    print('Model Performance on Train Dataset')
    print('Confusion Matrix:\n', train_conf_matrix)
    print('Classification Report:\n', train_class_report)    
    print('Accuracy:', train_accuracy)
    print('\n')
    print('----------------------------------')
    print('\n')
    print('Model Performance on Test Dataset')
    print('Confusion Matrix:\n', test_conf_matrix)
    print('Classification Report:\n', test_class_report)    
    print('Accuracy:', test_accuracy)
    
    accuracy_scores_list.append(test_accuracy)
    
    print('='*35)
    print('\n')
    
    

In [ ]:
pd.DataFrame(list(zip(models_list, accuracy_scores_list)), columns=['Model Name', 'Accuracy_Score']).sort_values(by=["Accuracy_Score"],ascending=False)

## 6. Model Evaluation

In [ ]:
# For this project I have trained multi models and I will use the best model to predict the data and all will be done in the above code


# conf_matrix = confusion_matrix(y_test, y_pred)
# class_report = classification_report(y_test, y_pred)
# accuracy = accuracy_score(y_test, y_pred)


# print("Confusion Matrix:")
# print(conf_matrix)
# print("\n Classification Report:")
# print(class_report)
# print("\n Accuracy:", accuracy)

## 7. Model Interpretation & Explainability     

#### 7.1 Feature Importance

In [ ]:
print("MODEL: Random Forest")
rf_model = list(models.values())[0]
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
names = [features[i] for i in indices]


plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.barh(range(X.shape[1]), importances[indices])
plt.yticks(range(X.shape[1]), names)
plt.show()

print('='*150)
print('\n')






print("MODEL: Gradient Boosting")
gb_model = list(models.values())[4]
importances = gb_model.feature_importances_
indices = np.argsort(importances)[::-1]
names = [features[i] for i in indices]


plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.barh(range(X.shape[1]), importances[indices])
plt.yticks(range(X.shape[1]), names)
plt.show()

print('='*150)
print('\n')








print("MODEL: KNN")
knn_model = list(models.values())[2]

# Calculate permutation importance
result = permutation_importance(knn_model, X_train, y_train, n_repeats=5, random_state=42)
importances = result.importances_mean
std = result.importances_std

# Sort features by importance
indices = np.argsort(importances)[::-1]
names = [features[i] for i in indices]

# Plot
plt.figure(figsize=(10, 6))
plt.title("Feature Importances (Permutation)")
plt.barh(range(len(importances)), importances[indices])
plt.yticks(range(len(importances)), names)
plt.xlabel("Permutation Importance")
plt.tight_layout()
plt.show()

print('='*150)
print('\n')









print("MODEL: SVM")
svm_model = list(models.values())[3]

result = permutation_importance(svm_model, X_test, y_test, n_repeats=5, random_state=42)
importances = result.importances_mean
std = result.importances_std
indices = np.argsort(importances)[::-1]
features = X_test.columns if hasattr(X_test, 'columns') else [f"feature {i}" for i in range(X_test.shape[1])]
names = [features[i] for i in indices]

plt.figure(figsize=(10, 6))
plt.title("SVM Feature Importance (Permutation)")
plt.barh(range(len(importances)), importances[indices])
plt.yticks(range(len(importances)), names)
plt.xlabel("Mean accuracy decrease")
plt.tight_layout()
plt.show()



### EXTRA: Re-building the model using feature engineering to add new features

In [13]:
df_1 = pd.read_csv('./data.csv')

# Binary feature for Balance
df_1['BalanceZero'] = (df_1['Balance'] == 0).astype(int)

# Age groups
df_1['AgeGroup'] = pd.cut(df_1['Age'], bins=[18, 25,35,45,55,65,75,85,95], labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66-75', '76-85', '86-95'])

# Balance to Salary ratio
df_1['BalanceToSalaryRatio'] = df_1['Balance'] / df_1['EstimatedSalary']

# Interaction feature between NumOfProducts and isActiveMember
df_1['ProductUsage'] = df_1['NumOfProducts'] * df_1['IsActiveMember']

# Tenure grouping
df_1['TenureGroup'] = pd.cut(df_1['Tenure'], bins=[0,2,5,7,10], labels=['0-2', '3-5', '6-7', '8-10'])

In [14]:
label_encoder = LabelEncoder()
df_1['Gender'] = label_encoder.fit_transform(df_1['Gender'])
df_1= pd.get_dummies(df_1, columns=['Geography'], drop_first=True)
df_1['Male_Germany'] = df_1['Gender'] * df_1['Geography_Germany']
df_1['Male_Spain'] = df_1['Gender'] * df_1['Geography_Spain']
df_1 = pd.get_dummies(df_1, columns=['AgeGroup', 'TenureGroup'], drop_first=True)

In [16]:
features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'BalanceZero', 'BalanceToSalaryRatio', 'ProductUsage', 'Male_Germany', 'Male_Spain'] + [col for col in df_1.columns if 'AgeGroup_' in col or 'TenureGroup_' in col]

X = df_1[features]
y = df_1['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


def model_evaluation(true, predicted):
    conf_matrix = confusion_matrix(true, predicted)
    class_report = classification_report(true, predicted)
    accuracy = accuracy_score(true, predicted)
    return conf_matrix, class_report, accuracy

train_conf_matrix, train_class_report, train_accuracy = model_evaluation(y_train, y_train_pred)
test_conf_matrix, test_class_report, test_accuracy = model_evaluation(y_test, y_test_pred)


print(f"Model: Random Forest")
print('Model Performance on Train Dataset')
print('Confusion Matrix:\n', train_conf_matrix)
print('Classification Report:\n', train_class_report)
print('Accuracy:', train_accuracy)

print('\n')
print('----------------------------------')
print('\n')

print('Model Performance on Test Dataset')
print('Confusion Matrix:\n', test_conf_matrix)
print('Classification Report:\n', test_class_report)
print('Accuracy:', test_accuracy)


Model: Random Forest
Model Performance on Train Dataset
Confusion Matrix:
 [[6356    0]
 [   0 1644]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6356
           1       1.00      1.00      1.00      1644

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000

Accuracy: 1.0


----------------------------------


Model Performance on Test Dataset
Confusion Matrix:
 [[1547   60]
 [ 210  183]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.96      0.92      1607
           1       0.75      0.47      0.58       393

    accuracy                           0.86      2000
   macro avg       0.82      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000

Accuracy: 0.865


## 8. Model Interpretation & Explainability    